In [4]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    GlobalMaxPooling1D,
    Bidirectional,
    Input
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [5]:
def negation_tagging(text):
    
    negation_words = ["not", "no", "never", "n't"]
    
    tokens = text.split()
    negated = False
    result = []
    
    for token in tokens:
        
        if token.lower() in negation_words:
            negated = True
            result.append(token)
            continue
        
        if negated:
            token = "NOT_" + token
        
        if re.search(r"[.!?]", token):
            negated = False
        
        result.append(token)
    
    return " ".join(result)

In [8]:
df = pd.read_csv("Suicide_Detection.csv")

texts = df["text"]
labels = df["class"]

In [9]:
texts = texts.apply(negation_tagging)

In [10]:
train_text, test_text, train_output, test_output = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)

In [11]:
max_vocab = 50000
max_len = 50

tokenizer = Tokenizer(num_words=max_vocab, oov_token="<OOV>")

tokenizer.fit_on_texts(train_text)

In [12]:
train_seq = tokenizer.texts_to_sequences(train_text)
test_seq = tokenizer.texts_to_sequences(test_text)

In [13]:
train_text_pad = pad_sequences(train_seq, maxlen=max_len)
test_text_pad = pad_sequences(test_seq, maxlen=max_len)

In [14]:
model = Sequential()

model.add(Input(shape=(max_len,)))

model.add(Embedding(
    input_dim=max_vocab,
    output_dim=100
))

model.add(Bidirectional(LSTM(64, return_sequences=True)))

model.add(GlobalMaxPooling1D())

model.add(Dense(128, activation='relu'))

model.add(Dense(1, activation='sigmoid'))

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [16]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

reducelr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=2
)

In [18]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

train_output = encoder.fit_transform(train_output)
test_output = encoder.transform(test_output)

In [19]:
print(train_output[:10])
print(type(train_output[0]))

[1 0 1 0 0 0 1 0 0 1]
<class 'numpy.int32'>


In [20]:
r = model.fit(
    train_text_pad,
    train_output,
    validation_data=(test_text_pad, test_output),
    epochs=20,
    batch_size=128,
    callbacks=[early_stop, reducelr]
)

Epoch 1/20
1451/1451 [==============================] - 414s 280ms/step - loss: 0.2409 - accuracy: 0.9032 - val_loss: 0.2053 - val_accuracy: 0.9196 - lr: 0.0010
Epoch 2/20
1451/1451 [==============================] - 441s 304ms/step - loss: 0.1784 - accuracy: 0.9318 - val_loss: 0.2065 - val_accuracy: 0.9182 - lr: 0.0010
Epoch 3/20
1451/1451 [==============================] - 426s 293ms/step - loss: 0.1469 - accuracy: 0.9454 - val_loss: 0.2221 - val_accuracy: 0.9172 - lr: 0.0010
Epoch 4/20
1451/1451 [==============================] - 396s 273ms/step - loss: 0.1006 - accuracy: 0.9642 - val_loss: 0.2418 - val_accuracy: 0.9156 - lr: 3.0000e-04


In [21]:
y_pred_prob = model.predict(test_text_pad)

y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(test_output, y_pred))

1451/1451 [==============================] - 49s 19ms/step
              precision    recall  f1-score   support

           0       0.93      0.91      0.92     23287
           1       0.91      0.93      0.92     23128

    accuracy                           0.92     46415
   macro avg       0.92      0.92      0.92     46415
weighted avg       0.92      0.92      0.92     46415



In [22]:
model.save("model.h5")

import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

c:\Users\user\anaconda3\envs\sd_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [28]:
twt = ["I am not suicidal"]
twt = tokenizer.texts_to_sequences(twt)
twt = pad_sequences(twt, maxlen=50)

prediction = model.predict(twt)[0][0]
print(prediction)

if prediction > 0.6:
    print("Potential Suicide Post")
else:
    print("Non Suicide Post")

1/1 [==============================] - 0s 67ms/step
0.96007156
Potential Suicide Post
